# FPP05 — Minutiae-Guided Early-Layer Attention (EfficientNetV2-S, incremental over the FPP04L winner; 3 repetitions)

**Question (one hypothesis):** does guiding the early layers of the image model towards **FingerJetFX minutiae during training** improve the **FPP04L winner** — above all **cross-sensor**? **Guidance is training-only:** the deployed encoder is unchanged (same architecture, size and speed, image only, cosine matching; no FingerJetFX at inference).

**Incremental, nothing repeated.** The previous winner is **read from the saved FPP04L outputs** (section 1b): FPP04L's EfficientNetV2-S candidates were the FPP04C model (re-scored) + `bce_cosface` + `bce_proxyanchor`; the same decision rule (lowest mean source-balanced validation EER, AUC breaks ties) picks the winner. That winner is (1) the **baseline** row here, **re-scored from its saved pair scores, not re-trained**, and (2) the **start weights** of every FPP05 run (same repetition). FPP05 keeps the winner's **whole configuration** (its loss, incl. the shared cross-sensor proxies if it is a proxy arm) and adds **only** the minutiae guidance.

| Candidate | Training loss on every batch (64 pairs) | Extra parameters |
|---|---|---|
| FPP04L winner (baseline) | — re-scored from its saved outputs, not re-trained | — |
| `mg_attention` | the winner's loss + λ · **attention transfer**: the activation-energy map `mean_c(F²)` of the stride-4 (56×56) and stride-8 (28×28) features is pulled towards the FingerJetFX minutia heatmap (+ small uniform floor) | none |
| `mg_attention_head` | the same + λ · **auxiliary heatmap head** (1×1 conv per guided layer, focal BCE vs the heatmap) | heads, **discarded after training** |

If the winner is a proxy arm, its finger proxies were discarded after FPP04L (they are not part of the model); they are re-initialised at the finger mean embeddings of the start encoder, exactly as FPP04L initialised them, and discarded again.

**Heatmap labels (FPM04D method):** FingerJetFX minutiae (FPP02 templates) mapped with the FPP02 letterbox geometry onto the 224×224 image, one Gaussian per minutia (σ 4 px, amplitude 0.3 + 0.7·quality, max-combined); alignment checked numerically and visually. A training image that is one of the FPP01 stored copies gets its heatmap warped with **the same stored rotation and shift**. Images without minutiae get no guidance.

**Training:** full fine-tuning (BatchNorm frozen), AdamW 1e-5; λ cosine 0.5 → 0.05 over the first 40 % of epochs, then 0.05; ≤ 50 epochs, patience 10 / LR × 0.3 after 3; batch 64 pairs, source-balanced sampler, FPP01 materialized augmentation.

**Decision (validation only):** candidates = FPP04L winner + the two arms; **winner = lowest mean source-balanced validation EER** (AUC breaks ties). Selection table: validation only — balanced and DB1 / DB2 / DB1×DB2: EER, AUC, F1, FMR, FNMR (mean ± std) — plus training time. **Test (MOLF balanced, DB1×DB2) and FingerDOS are measured and reported for every candidate but are not used for the decision.** Bias check with the 5 calibration subjects; attention diagnostic on validation images.

**Time:** the inference network is identical to FPP04C / FPP04L; GPU embedding time is measured per run; the fair 1-thread CPU comparison with FingerJetFX + BOZORTH3 (FPP04C cpu_timing) applies unchanged and is shown in section 6.

**Hardware:** A100, High-RAM; 2 arms × 3 repetitions = 6 runs (≈ 3 h). **Needs:** FPP01, FPP02 (images + templates), FPP04C and **FPP04L finished**, fpp_lib ≥ 1.5.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared libraries from output/00_code/ (fpf_lib from FPF00, fpp_lib installed by FPP01), so every FPP kernel uses the same version.
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpp_lib.py'):
    src = CODE / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run ' + ('FPP01' if m == 'fpp_lib.py' else 'FPF00') + ' first')
    shutil.copy2(src, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpf_tf as T; importlib.reload(T)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
if tuple(int(x) for x in P.VERSION.split('.')) < (1, 5): raise RuntimeError(f'fpp_lib {P.VERSION} in output/00_code is too old: upload fpp_lib 1.5')
ENV = L.env_info(); print('fpp_lib', P.VERSION, '|', json.dumps(ENV, indent=1))

## 1. Configuration

In [ ]:
import h5py, subprocess, math, tensorflow as tf
import matplotlib.pyplot as plt
KERNEL = 'FPP05'; PREV_KERNEL = 'FPP04L'; MODEL_NAME = 'EfficientNetV2S'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'
REPS = L.repetitions(ws); REPS_TO_RUN = REPS
ARMS = {'mg_attention': dict(at=True, head=False), 'mg_attention_head': dict(at=True, head=True)}
BACKBONES = list(ARMS)                                   # run names
PREVIOUS_WINNER = None                                   # None -> read from the saved FPP04L outputs by the decision rule (section 1b); never re-trained
LAMBDA_PROXY = 1.0; COSFACE_S = 30.0; COSFACE_M = 0.35; PA_ALPHA = 32.0; PA_DELTA = 0.1; PROXY_NORM = 0.01   # FPP04L values (used only if the winner is a proxy arm)
HEATMAP_SIGMA = 4.0; QUALITY_FLOOR = 0.3; FOREGROUND_GRAY = 240; GUIDED_SIZES = (56, 28)
LAMBDA_START = 0.5; LAMBDA_END = 0.05; LAMBDA_DECAY_FRACTION = 0.4; AT_TARGET_FLOOR = 0.10; FOCAL_GAMMA = 2.0; HEAD_BIAS_INIT = -2.0; HEAD_LR_MULT = 100.0
ATTENTION_CHECK_IMAGES = 256; INIT_CHECK_MIN = 0.99
ARMS_TO_RUN = BACKBONES            # e.g. ['mg_attention'] to split the work across sessions
EMBEDDING_DIM = 1024; IMAGE_SIZE = 224; DROPOUT = 0.30
LR_FULL = 1e-5; WEIGHT_DECAY = 1e-4; EPOCHS = 50; PATIENCE = 10; LR_PATIENCE = 3; LR_FACTOR = 0.3
PAIR_BATCH = 64; TRAIN_PAIRS_PER_EPOCH = 12000; VAL_PAIRS = None; EMBED_BATCH = 256     # A100; VAL_PAIRS caps the validation pairs per set (smoke test only)
JIT_COMPILE = False   # XLA: often 10-30 % faster on A100; set True per session if every backbone compiles
VIT_PRESET = 'vit_base_patch16_224_imagenet'; PARTIAL_LAST_FRACTION_FALLBACK = 0.20
TIME_CPU = False; TIMING_REPEATS = 30
TRAIN_AUGMENTATION = True; AUG_P_ORIGINAL = 0.25     # training images: original with p 0.25, else one of the 3 FPP01 copies (uniform)
QUICK_SMOKE_TEST = False
if QUICK_SMOKE_TEST:
    EPOCHS = 1; TRAIN_PAIRS_PER_EPOCH = 384; VAL_PAIRS = 384; ARMS_TO_RUN = ['mg_attention_head']; REPS_TO_RUN = REPS[:1]; KDIR = KDIR / '_smoke_test'; FIG = KDIR / 'figures'
FIG.mkdir(parents=True, exist_ok=True); (KDIR / 'runs').mkdir(parents=True, exist_ok=True); LOG = L.RunLog(KDIR / 'FPP05_run_log.csv')
MIXED = T.use_mixed_precision()
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], protocol=P.PROTOCOL_TABLE, backbone=MODEL_NAME, arms=ARMS, baseline='FPP04L winner (re-scored from its saved outputs, not re-trained)', decision=P.DECISION_RULE, policy='full (all layers except BatchNorm), initialised from the FPP04L winner, same loss + minutiae guidance', previous_kernel=PREV_KERNEL, heatmap=f'letterbox-aligned Gaussian sigma {HEATMAP_SIGMA}px, amplitude {QUALITY_FLOOR}+{1 - QUALITY_FLOOR}*quality, max-combined; stored copies: same stored rotation/shift', guided_feature_sizes=GUIDED_SIZES, at_loss=f'||l2n(mean_c F^2) - l2n(avgpool(H) + {AT_TARGET_FLOOR})||^2', head_loss=f'1x1 conv, focal BCE gamma {FOCAL_GAMMA} vs maxpool(H), normalised by heatmap mass; discarded', lambda_schedule=f'cosine {LAMBDA_START} -> {LAMBDA_END} over the first {int(100 * LAMBDA_DECAY_FRACTION)}% of epochs, then {LAMBDA_END}', embedding_dim=EMBEDDING_DIM, image_size=IMAGE_SIZE, dropout=DROPOUT,
              loss='cosine-BCE (learned calibration)', lr=LR_FULL, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
              pair_batch=PAIR_BATCH, train_pairs_per_epoch=TRAIN_PAIRS_PER_EPOCH, sampler='source-balanced DB1 / DB2 / DB1xDB2', validation='all DB1, DB2, DB1xDB2 validation pairs, embedded once per epoch', early_stopping=P.SELECTION_RULE,
              local_runs=str(P.LOCAL_RUNS), embed_batch=EMBED_BATCH, jit_compile=JIT_COMPILE,
              batchnorm='frozen', mixed_precision=MIXED, training_augmentation=(f'materialized FPP01 copies x{P.AUG_COPIES}, original with p {AUG_P_ORIGINAL}' if TRAIN_AUGMENTATION else 'none'),
              augmentation_recipe=P.AUG_RECIPE if TRAIN_AUGMENTATION else None, threshold=P.THRESHOLD_RULE, metrics=P.METRICS, ranking=f'mean {P.SELECTION_SET} EER over repetitions (AUC breaks ties)', vit_preset=VIT_PRESET,
              repetitions=REPS, quick_smoke_test=QUICK_SMOKE_TEST, fpp_lib_version=P.VERSION)
print(json.dumps(CONFIG, indent=2, default=str))

## 1b. Previous winner: read from the saved FPP04L outputs (validation only, not re-run)

In [ ]:
# FPP04L EfficientNetV2-S candidates (as decided there): the FPP04C model (re-scored) + the two shared-proxy losses; bce_control was a check, not a candidate
FPP04C_DIR = Path(ws['output']) / P.FPP_KERNELS['FPP04C']; FPP04L_DIR = Path(ws['output']) / P.FPP_KERNELS[PREV_KERNEL]
PREV = {f'{MODEL_NAME}__FPP04C': (FPP04C_DIR, MODEL_NAME, 'bce'), **{f'{MODEL_NAME}__{l}': (FPP04L_DIR, f'{MODEL_NAME}__{l}', l) for l in ('bce_cosface', 'bce_proxyanchor')}}
PREV_ROWS = []
for lab, (kd, arm, _) in PREV.items():
    rr = P.reference_rows(lab, kd, arm=arm, reps=REPS)
    if len({r_['repetition'] for r_ in rr}) != len(REPS): raise FileNotFoundError(f'{lab}: saved runs not found in {kd.name} (finish FPP04C / FPP04L first)')
    for r_ in rr: r_['kernel'] = lab
    PREV_ROWS += rr
prev_results = pd.DataFrame(PREV_ROWS); prev_winner, prev_decision = P.decide(prev_results, list(PREV))
print('FPP04L decision (re-computed from its saved outputs, validation only):'); display(prev_decision.round(3)); display(P.validation_table(prev_results, list(PREV)))
if PREVIOUS_WINNER is not None:
    if PREVIOUS_WINNER not in PREV: raise ValueError(f'PREVIOUS_WINNER must be one of {list(PREV)}')
    print(f'NOTE: PREVIOUS_WINNER forced to {PREVIOUS_WINNER} (the rule gives {prev_winner})'); prev_winner = PREVIOUS_WINNER
BASE_DIR, BASE_ARM, BASE_LOSS = PREV[prev_winner]; BASELINE = f"{PREV_KERNEL} winner ({prev_winner.split('__')[1]})"
def base_run(rep): return BASE_DIR / 'runs' / rep / BASE_ARM
CONFIG.update(previous_winner=prev_winner, baseline_label=BASELINE, start_weights=L.rel(ws, base_run(REPS[0])).replace(REPS[0], '<rep>'), training_loss=f'{BASE_LOSS} + minutiae guidance')
print(f'BASELINE = {BASELINE} | start weights: {BASE_ARM} from {BASE_DIR.name} | training loss: {BASE_LOSS} + minutiae guidance')

## 2. Images (FPP02), protocol and augmented copies (FPP01), minutia heatmaps (FPP02 templates)

In [ ]:
IMG = {}
for d in L.DATASETS:
    with h5py.File(L.local_copy(ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5'), 'r') as h:
        if not h['complete'][:].all(): raise RuntimeError(f'{d}: image cache incomplete; finish FPF00')
        IMG[d] = h['images'][:]
    print(d, IMG[d].shape)
MOLF = ('MOLF_DB1', 'MOLF_DB2'); IMG_MOLF = np.concatenate([IMG[d] for d in MOLF]); OFFSET = {'MOLF_DB1': 0, 'MOLF_DB2': len(IMG['MOLF_DB1'])}
PROTO = {r: P.load_pairs(ws, r) for r in REPS}; print('FPP01 pair files verified for', len(REPS), 'repetitions')
AUG_MOLF = None
if TRAIN_AUGMENTATION:
    AUG = {}; AUGP = {}
    for d in MOLF:
        f = P.augmentation_file(ws, d)
        if not f.is_file(): raise FileNotFoundError(f'{f} missing: run FPP01 section 6 first')
        with h5py.File(L.local_copy(f), 'r') as h:
            if not h['complete'][:].all(): raise RuntimeError(f'{d}: augmentation incomplete; finish FPP01')
            if h.attrs['source_sha256'] != L.sha256_of(L.local_copy(ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5')): raise RuntimeError(f'{d}: augmented copies were made from a different image cache')
            AUG[d] = h['images'][:]; AUGP[d] = h['params'][:]                      # per copy: angle, shift_x, shift_y, zoom, brightness, contrast
        print(d, 'augmented copies', AUG[d].shape)
    AUG_MOLF = np.concatenate([AUG[d] for d in MOLF]); AUGP_MOLF = np.concatenate([AUGP[d] for d in MOLF]); del AUG                      # same index space as IMG_MOLF: (image, copy, H, W, 1)
class PairSeq(tf.keras.utils.Sequence):
    # Keras view of the fpp_lib source-balanced sampler; training: each image original (p AUG_P_ORIGINAL) or one of its stored copies, reproducible per (seed, epoch, batch)
    def __init__(self, images, sampler, aug=None, p_original=1.0, seed=0, workers=4):
        super().__init__(workers=workers, use_multiprocessing=False, max_queue_size=8); self.images = images; self.s = sampler; self.aug = aug; self.p = p_original; self.seed = seed; self.epoch = 0
    def __len__(self): return len(self.s)
    def __getitem__(self, b):
        ia, ib, y = self.s.batch(b)
        if self.aug is None: return (self.images[ia], self.images[ib]), y
        rng = np.random.default_rng([self.seed, self.epoch, b]); return (P.mix_versions(self.images, self.aug, ia, self.p, rng), P.mix_versions(self.images, self.aug, ib, self.p, rng)), y
    def on_epoch_end(self): self.s.on_epoch_end(); self.epoch += 1
# ---- FingerJetFX minutia heatmaps on the 224x224 letterbox (training labels only)
import cv2
def letterbox_geometry(w, h, size=IMAGE_SIZE):
    s = min(size / w, size / h); nw = max(1, int(round(w * s))); nh = max(1, int(round(h * s))); return nw / w, nh / h, (size - nw) // 2, (size - nh) // 2
def render_heatmap(px, py, q, size=IMAGE_SIZE, sigma=HEATMAP_SIGMA):
    H = np.zeros((size, size), np.float32); r = int(math.ceil(3 * sigma)); amp = QUALITY_FLOOR + (1.0 - QUALITY_FLOOR) * q
    for cx, cy, a in zip(px, py, amp):
        if not (np.isfinite(cx) and np.isfinite(cy)): continue
        x0 = max(0, int(math.floor(cx)) - r); x1 = min(size, int(math.floor(cx)) + r + 2); y0 = max(0, int(math.floor(cy)) - r); y1 = min(size, int(math.floor(cy)) + r + 2)
        if x0 >= x1 or y0 >= y1: continue
        gx = np.arange(x0, x1, dtype=np.float32) - cx; gy = np.arange(y0, y1, dtype=np.float32) - cy
        H[y0:y1, x0:x1] = np.maximum(H[y0:y1, x0:x1], a * np.exp(-(gy[:, None] ** 2 + gx[None, :] ** 2) / (2 * sigma * sigma)))
    return H
TPL = {}; HM = {}; HAS_MIN = {}; LB = {}
for d in L.DATASETS:
    with h5py.File(L.local_copy(ws['templates'] / f'{d}_fingerjetfx_templates.h5'), 'r') as h:
        g = h['minutiae']; TPL[d] = {k: g[k][:] for k in ('x', 'y', 'quality', 'image_width', 'image_height')}
    n = len(IMG[d]); assert len(TPL[d]['x']) == n, f'{d}: templates and images differ in count'
    HM[d] = np.zeros((n, IMAGE_SIZE, IMAGE_SIZE, 1), np.uint8); LB[d] = []
    for i in range(n):
        x = np.asarray(TPL[d]['x'][i], np.float32); y = np.asarray(TPL[d]['y'][i], np.float32); q = np.clip(np.asarray(TPL[d]['quality'][i], np.float32), 0, 1)
        sx, sy, ox, oy = letterbox_geometry(int(TPL[d]['image_width'][i]), int(TPL[d]['image_height'][i])); px = (x + 0.5) * sx - 0.5 + ox; py = (y + 0.5) * sy - 0.5 + oy
        LB[d].append((px, py)); HM[d][i, ..., 0] = np.rint(255 * render_heatmap(px, py, q)).astype(np.uint8)
    HAS_MIN[d] = np.array([len(v) > 0 for v in TPL[d]['x']]); print(d, 'heatmaps', HM[d].shape, f'| images with >=1 minutia {100 * HAS_MIN[d].mean():.2f} %')
HM_MOLF = np.concatenate([HM[d] for d in MOLF]); HASMIN_MOLF = np.concatenate([HAS_MIN[d] for d in MOLF]).astype(np.float32)
def warp_like_copy(hm, p):
    # the stored copy's geometric transform (augment_image: rotation about the centre, zoom, shift) applied to the heatmap (fill 0 = no minutia)
    M = cv2.getRotationMatrix2D((IMAGE_SIZE / 2.0, IMAGE_SIZE / 2.0), float(p[0]), float(p[3])); M[0, 2] += float(p[1]); M[1, 2] += float(p[2])
    return cv2.warpAffine(hm[..., 0], M, (IMAGE_SIZE, IMAGE_SIZE), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)[..., None]
HM_AUG = None
if AUG_MOLF is not None:
    HM_AUG = np.zeros(AUG_MOLF.shape, np.uint8)
    for i in range(len(HM_MOLF)):
        for k in range(AUG_MOLF.shape[1]): HM_AUG[i, k] = warp_like_copy(HM_MOLF[i], AUGP_MOLF[i, k])
    print('heatmaps of the stored copies', HM_AUG.shape)

# ---- alignment checks: minutiae inside the canvas and on the print (vs a point-flipped control), on originals and on stored copies
ALIGN = []
for d in L.DATASETS:
    idx = np.random.default_rng(0).choice(len(IMG[d]), min(1000, len(IMG[d])), replace=False); tot = ins = onp = onp_flip = 0
    for i in idx:
        px, py = LB[d][i]
        if len(px) == 0: continue
        im = IMG[d][i, ..., 0].astype(np.float32); tot += len(px)
        for cx, cy, flip in ((px, py, False), (IMAGE_SIZE - 1 - px, IMAGE_SIZE - 1 - py, True)):
            ok = (cx >= 0) & (cx < IMAGE_SIZE) & (cy >= 0) & (cy < IMAGE_SIZE)
            if not flip: ins += int(ok.sum())
            hits = sum(im[max(0, b - 2):b + 3, max(0, a - 2):a + 3].mean() < FOREGROUND_GRAY for a, b in zip(cx[ok].astype(int), cy[ok].astype(int)))
            if flip: onp_flip += hits
            else: onp += hits
    ALIGN.append({'dataset': d, 'minutiae_checked': tot, 'inside_canvas_pct': 100 * ins / max(tot, 1), 'on_fingerprint_pct': 100 * onp / max(tot, 1), 'on_fingerprint_pct_flipped_control': 100 * onp_flip / max(tot, 1)})
if HM_AUG is not None:                                   # copies: heat mass must sit on the warped print as much as on the original
    j = np.random.default_rng(1).choice(len(HM_MOLF), 300, replace=False); fg = lambda im, hm: float((hm[..., 0].astype(np.float32) * (im[..., 0] < FOREGROUND_GRAY)).sum() / max(hm.sum(), 1))
    ALIGN.append({'dataset': 'MOLF stored copies', 'minutiae_checked': len(j), 'on_fingerprint_pct': 100 * np.mean([fg(AUG_MOLF[i, 0], HM_AUG[i, 0]) for i in j]),
                  'on_fingerprint_pct_flipped_control': 100 * np.mean([fg(AUG_MOLF[i, 0], HM_AUG[i, 0][::-1, ::-1]) for i in j]), 'inside_canvas_pct': np.nan})
alignment = pd.DataFrame(ALIGN); display(alignment.round(2))
if (alignment.on_fingerprint_pct < 85).any(): print('WARNING: minutiae do not line up well with the images; inspect the overlay before training')
fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for c_, i in enumerate(np.random.default_rng(3).choice(len(IMG['MOLF_DB1']), 2, replace=False)):
    px, py = LB['MOLF_DB1'][i]; a = axs[0, 2 * c_]; a.imshow(IMG['MOLF_DB1'][i, ..., 0], cmap='gray', vmin=0, vmax=255); a.imshow(HM['MOLF_DB1'][i, ..., 0], cmap='jet', alpha=.35); a.plot(px, py, 'r+', ms=4); a.set_title(f'DB1 {i} original', fontsize=8); a.axis('off')
    if HM_AUG is not None:
        a = axs[0, 2 * c_ + 1]; a.imshow(AUG_MOLF[i, 0, ..., 0], cmap='gray', vmin=0, vmax=255); a.imshow(HM_AUG[i, 0, ..., 0], cmap='jet', alpha=.35); a.set_title(f'DB1 {i} stored copy 0', fontsize=8); a.axis('off')
for c_, d in enumerate(('MOLF_DB2', 'FingerDOS', 'MOLF_DB2', 'FingerDOS')):
    i = int(np.random.default_rng(10 + c_).integers(len(IMG[d]))); px, py = LB[d][i]; a = axs[1, c_]
    a.imshow(IMG[d][i, ..., 0], cmap='gray', vmin=0, vmax=255); a.imshow(HM[d][i, ..., 0], cmap='jet', alpha=.35); a.plot(px, py, 'r+', ms=4); a.set_title(f'{d} {i}', fontsize=8); a.axis('off')
fig.suptitle('FingerJetFX minutiae (+) and heatmap labels on the 224x224 letterbox (stored copies: same stored transform)', fontsize=10); P._save(fig, FIG / 'heatmap_alignment_overlay.png')

# ---- shared cross-sensor finger classes (FPP04L code) — only needed when the FPP04L winner is a proxy arm
def finger_classes(proto):
    lab = np.full(len(IMG_MOLF), -1, np.int64); keys = {}
    for d in MOLF:
        s = proto['samples'][d]; t_ = s[s.partition == 'train']
        for pos, key in zip(t_.sample_index.to_numpy(), t_.identity_id.astype(str).to_numpy()): lab[OFFSET[d] + int(pos)] = keys.setdefault(key, len(keys))
    sens = {k: set() for k in range(len(keys))}
    for d in MOLF:
        for k in np.unique(lab[OFFSET[d]:OFFSET[d] + len(IMG[d])]):
            if k >= 0: sens[int(k)].add(d)
    if sum(len(v) == 2 for v in sens.values()) != len(keys): raise RuntimeError('some training fingers lack one sensor: proxies would not be shared')
    return lab, len(keys)
CLASSES = {r: finger_classes(PROTO[r][0]) for r in REPS} if BASE_LOSS != 'bce' else {r: (np.zeros(len(IMG_MOLF), np.int64), 0) for r in REPS}
if BASE_LOSS != 'bce': print({r: f'{k} shared finger proxies' for r, (_, k) in CLASSES.items()})

class GuidedPairSeq(PairSeq):
    # PairSeq + the heatmap of exactly the image version shown (original or stored copy k), a has-minutiae weight and the shared finger class
    def __init__(self, labels, *a, **k): super().__init__(*a, **k); self.labels = labels
    def __getitem__(self, b):
        ia, ib, y = self.s.batch(b); rng = np.random.default_rng([self.seed, self.epoch, b]); out = []
        for idx in (ia, ib):
            idx = np.asarray(idx); x = np.array(self.images[idx], copy=True); h = np.array(HM_MOLF[idx], copy=True)
            if self.aug is not None and self.p < 1:
                use = rng.random(len(idx)) >= self.p; k = rng.integers(0, self.aug.shape[1], len(idx))       # identical draw order to P.mix_versions
                if use.any(): x[use] = self.aug[idx[use], k[use]]; h[use] = HM_AUG[idx[use], k[use]]
            out.append((x, h, HASMIN_MOLF[idx]))
        (xa, ha, wa), (xb, hb, wb) = out; return (xa, xb, ha, hb, wa, wb, self.labels[ia], self.labels[ib]), y

## 3. Encoder (FPM04 code, `full` policy), guided Siamese model, balanced-EER callback

In [ ]:
A = tf.keras.applications
RULES = {'DenseNet201': ('prefix', ('conv5_',)), 'ResNet50': ('prefix', ('conv5_',)), 'VGG16': ('prefix', ('block5_',)), 'InceptionV3': ('after', 'mixed8'),
         'EfficientNetV2S': ('prefix', ('block6', 'top_')), 'MobileNetV3Large': ('prefix', ('expanded_conv_12', 'expanded_conv_13', 'expanded_conv_14', 'conv_1', 'Conv_1')),
         'ConvNeXtTiny': ('contains', ('stage_3', 'downsampling_block_2')), 'ViTB16': ('last_fraction', PARTIAL_LAST_FRACTION_FALLBACK)}

def make_backbone(name):
    # Returns (backbone model: (224,224,3) float 0..255 -> feature vector, preprocessing function or None)
    shape = (IMAGE_SIZE, IMAGE_SIZE, 3); kw = dict(include_top=False, weights='imagenet', input_shape=shape, pooling='avg')
    if name == 'DenseNet201': return A.DenseNet201(**kw), A.densenet.preprocess_input
    if name == 'ResNet50': return A.ResNet50(**kw), A.resnet50.preprocess_input
    if name == 'VGG16': return A.VGG16(**kw), A.vgg16.preprocess_input
    if name == 'InceptionV3': return A.InceptionV3(**kw), A.inception_v3.preprocess_input
    if name == 'EfficientNetV2S': return A.EfficientNetV2S(**kw), None                      # built-in preprocessing (expects 0..255)
    if name == 'MobileNetV3Large': return A.MobileNetV3Large(**kw), None                    # built-in preprocessing (expects 0..255)
    if name == 'ConvNeXtTiny': return A.ConvNeXtTiny(**kw), None                            # built-in preprocessing (expects 0..255)
    if name == 'ViTB16':
        try: import keras_hub
        except ImportError:                                                      # installed only when needed, without touching TensorFlow/Keras
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'keras-hub', 'kagglehub'], check=True); import keras_hub
        core = keras_hub.models.ViTBackbone.from_preset(VIT_PRESET)
        try: conv = keras_hub.layers.ViTImageConverter.from_preset(VIT_PRESET); pre = lambda z: conv(z)
        except Exception as exc: print('ViT image converter unavailable, using [-1, 1] scaling:', exc); pre = lambda z: z / 127.5 - 1.0
        inp = tf.keras.Input(shape); seq = core(pre(inp)); feat = tf.keras.layers.Lambda(lambda z: z[:, 0, :], name='cls_token')(seq)
        return tf.keras.Model(inp, feat, name='vit_b16_backbone'), None
    raise ValueError(name)

def leaf_layers(model):
    try: allv = list(model._flatten_layers(include_self=False, recursive=True))
    except AttributeError: allv = list(model.layers)
    out = []
    for l in allv:
        try: kids = list(l._flatten_layers(include_self=False, recursive=False))
        except AttributeError: kids = []
        if not kids: out.append(l)
    return out

def apply_policy(bb, name, policy):
    # 'full': everything trainable except BatchNorm. 'partial': only the layers named by RULES (BatchNorm always frozen).
    bb.trainable = True; leaves = leaf_layers(bb); rule, arg = RULES[name]; used = rule
    if policy == 'partial':
        if rule == 'prefix': sel = [l.name.startswith(arg) for l in leaves]
        elif rule == 'contains': sel = [any(a in l.name for a in arg) for l in leaves]
        elif rule == 'after':
            names = [l.name for l in leaves]; k = names.index(arg) if arg in names else len(names); sel = [i > k for i in range(len(names))]
        else: sel = [False] * len(leaves)
        weighted = [i for i, l in enumerate(leaves) if l.weights]
        if not any(sel[i] for i in weighted):
            used = f'last_fraction {PARTIAL_LAST_FRACTION_FALLBACK}' if rule == 'last_fraction' else f'fallback last_fraction {PARTIAL_LAST_FRACTION_FALLBACK} (rule matched nothing)'
            keep = set(weighted[int(len(weighted) * (1 - PARTIAL_LAST_FRACTION_FALLBACK)):]); sel = [i in keep for i in range(len(leaves))]
        for l, s in zip(leaves, sel): l.trainable = bool(s)
    for l in leaves:
        if isinstance(l, tf.keras.layers.BatchNormalization): l.trainable = False
    unfrozen = [l.name for l in leaves if l.trainable and l.weights]
    tr = int(sum(np.prod(w.shape) for w in bb.trainable_weights)); tot = int(sum(np.prod(w.shape) for w in bb.weights))
    return {'policy': policy, 'rule': used, 'unfrozen_layers': len(unfrozen), 'first_unfrozen': unfrozen[0] if unfrozen else '', 'last_unfrozen': unfrozen[-1] if unfrozen else '',
            'backbone_params_trainable': tr, 'backbone_params_total': tot}

def build_encoder(name, policy, emb_dim=EMBEDDING_DIM):
    bb, pre = make_backbone(name); info = apply_policy(bb, name, policy)
    image = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 1), dtype=tf.uint8, name='fingerprint_uint8')
    x = tf.keras.layers.Lambda(lambda z: tf.cast(z, tf.float32), name='cast_float')(image); x = tf.keras.layers.Concatenate(name='gray_to_rgb')([x, x, x])
    if pre is not None: x = tf.keras.layers.Lambda(pre, name='imagenet_preprocess')(x)
    f = bb(x, training=False); f = tf.keras.layers.Dropout(DROPOUT, name='head_dropout')(f)
    e = tf.keras.layers.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = tf.keras.layers.BatchNormalization(name='embedding_bn')(e)
    e = tf.keras.layers.Lambda(lambda z: tf.math.l2_normalize(tf.cast(z, tf.float32), axis=1), name='embedding')(e)
    return tf.keras.Model(image, e, name=f'{name}_{policy}_encoder'), info

def build_siamese_model(encoder, lr):
    shape = (IMAGE_SIZE, IMAGE_SIZE, 1); ia = tf.keras.Input(shape, dtype=tf.uint8, name='input_a'); ib = tf.keras.Input(shape, dtype=tf.uint8, name='input_b')
    cos = tf.keras.layers.Dot(axes=1, name='cosine_similarity')([encoder(ia), encoder(ib)])
    model = tf.keras.Model([ia, ib], T.CosineCalibration(name='same_finger_probability')(cos), name='siamese')
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY), loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='verification_accuracy'), tf.keras.metrics.AUC(name='verification_auc')], jit_compile=JIT_COMPILE)
    return model
class EpochClock(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['epoch_seconds'] = time.perf_counter() - self.t0
class BalancedEERCallback(tf.keras.callbacks.Callback):
    # after each epoch: embed the validation images, score all validation pairs, source-balanced EER -> best checkpoint / LR drop / early stop (fpp_lib rule)
    def __init__(self, encoder, bv, ckpt, hist_csv):
        super().__init__(); self.encoder = encoder; self.bv = bv; self.ckpt = ckpt; self.hist_csv = hist_csv; self.x = IMG_MOLF[bv.indices]; self.rows = []
        self.stop = P.BalancedEERStopping(PATIENCE, LR_PATIENCE, LR_FACTOR)
    def _lr(self): return float(np.asarray(self.model.optimizer.learning_rate))
    def _set_lr(self, v):
        try: self.model.optimizer.learning_rate.assign(v)
        except AttributeError: self.model.optimizer.learning_rate = v
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}; t0 = time.perf_counter()
        m = self.bv.score(L.l2n(T.embed(self.encoder, self.x, EMBED_BATCH))); logs.update(m); lr = self._lr(); d = self.stop.update(epoch, m['val_balanced_eer'], m['val_balanced_auc'])
        if d['improved']: self.model.save_weights(str(self.ckpt))
        if d['reduce_lr'] and lr > self.stop.min_lr: self._set_lr(max(lr * LR_FACTOR, self.stop.min_lr))
        if d['stop']: self.model.stop_training = True
        logs['validation_seconds'] = time.perf_counter() - t0
        self.rows.append({'epoch': epoch + 1, **{k: float(np.asarray(v)) for k, v in logs.items() if np.asarray(v).size == 1}, 'lr': lr, 'improved': d['improved']}); pd.DataFrame(self.rows).to_csv(self.hist_csv, index=False)
        print(f"  epoch {epoch + 1}: val balanced EER {100 * m['val_balanced_eer']:.2f} % | AUC {100 * m['val_balanced_auc']:.2f} % | "
              + ' '.join(f"{n.split('_')[0]} {100 * m[f'val_eer_{n}']:.2f}" for n in P.VAL_SOURCES) + (' *best' if d['improved'] else '') + (' | LR down' if d['reduce_lr'] else '') + (' | stop' if d['stop'] else ''))
def guided_layer_names(bb):
    # deepest layer at each guided spatial size (stride 4 / stride 8); a residual Add output is preferred when present (FPM04D)
    found = {}
    for l in bb.layers:
        try: s = tuple(l.output.shape)
        except Exception: continue
        if len(s) == 4 and s[1] in GUIDED_SIZES and s[1] == s[2]:
            prev = found.get(s[1])
            if prev is None or isinstance(l, tf.keras.layers.Add) or not prev[1]: found[s[1]] = (l.name, isinstance(l, tf.keras.layers.Add))
    missing = [s for s in GUIDED_SIZES if s not in found]
    if missing: raise RuntimeError(f'no {missing} feature map in {bb.name}')
    return [found[s][0] for s in GUIDED_SIZES]

class HeatHead(tf.keras.layers.Layer):
    # 1x1 conv to a minutia-heatmap logit; parameters scaled by HEAD_LR_MULT so the head learns ~100x faster than the encoder with one optimizer
    def build(self, shape):
        self.k = self.add_weight(name='k', shape=(int(shape[-1]), 1), initializer=tf.keras.initializers.RandomNormal(stddev=0.01 / HEAD_LR_MULT), dtype='float32')
        self.b = self.add_weight(name='b', shape=(1,), initializer=tf.keras.initializers.Constant(HEAD_BIAS_INIT / HEAD_LR_MULT), dtype='float32')
    def call(self, F): return HEAD_LR_MULT * (tf.tensordot(tf.cast(F, tf.float32), self.k, axes=[[3], [0]]) + self.b)

def energy(F): return tf.reduce_mean(tf.square(tf.cast(F, tf.float32)), axis=-1)
def pooled(H, F):
    st = IMAGE_SIZE // int(F.shape[1]); return tf.nn.avg_pool2d(H, st, st, 'VALID'), tf.nn.max_pool2d(H, st, st, 'VALID')
def at_loss(F, Ha, w):
    B_ = tf.shape(F)[0]; a = tf.math.l2_normalize(tf.reshape(energy(F), [B_, -1]), 1); t = tf.math.l2_normalize(tf.reshape(Ha, [B_, -1]) + AT_TARGET_FLOOR, 1)
    return tf.reduce_sum(tf.reduce_sum(tf.square(a - t), 1) * w) / tf.maximum(tf.reduce_sum(w), 1.0)
def head_loss(logits, Hm, w):
    p = tf.sigmoid(logits); ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=Hm, logits=logits); pt = Hm * p + (1 - Hm) * (1 - p)
    per = tf.reduce_sum(ce * tf.pow(1.0 - pt, FOCAL_GAMMA), [1, 2, 3]) / tf.maximum(tf.reduce_sum(Hm, [1, 2, 3]), 1.0)
    return tf.reduce_sum(per * w) / tf.maximum(tf.reduce_sum(w), 1.0)
def guidance_lambda(epoch):     # epoch counted from 1
    decay = max(1, int(round(LAMBDA_DECAY_FRACTION * EPOCHS))); t_ = min(1.0, (epoch - 1) / decay)
    return LAMBDA_END + (LAMBDA_START - LAMBDA_END) * 0.5 * (1 + math.cos(math.pi * t_))

class GuidedSiamese(tf.keras.Model):
    # shares every weight with the deployed encoder; loss = the FPP04L winner's loss (cosine-BCE [+ shared-proxy loss]) + lambda * (attention transfer [+ heatmap head])
    def __init__(self, encoder, spec, n_classes=0, init_proxies=None, **kw):
        super().__init__(**kw); self.encoder = encoder; self.spec = spec
        self.bb = [l for l in encoder.layers if isinstance(l, tf.keras.Model)][0]; self.guided = guided_layer_names(self.bb)
        self.multi = tf.keras.Model(self.bb.input, [self.bb.get_layer(n).output for n in self.guided] + [self.bb.output], name='backbone_multi_output')
        names = [l.name for l in encoder.layers]; self.pre = encoder.get_layer('imagenet_preprocess') if 'imagenet_preprocess' in names else None
        self.drop = encoder.get_layer('head_dropout'); self.dense = encoder.get_layer('embedding_dense'); self.ebn = encoder.get_layer('embedding_bn')
        self.heads = [HeatHead(name=f'aux_head_{s}') for s in GUIDED_SIZES] if spec['head'] else []
        self.calib = T.CosineCalibration(name='same_finger_probability'); self.lam = tf.Variable(LAMBDA_START, trainable=False, dtype=tf.float32, name='guidance_lambda')
        self.bce = tf.keras.losses.BinaryCrossentropy(); self.kind = BASE_LOSS; self.n_classes = n_classes
        if self.kind != 'bce':
            self.proxies = self.add_weight(name='finger_proxies', shape=(n_classes, EMBEDDING_DIM), dtype='float32', trainable=True, initializer='zeros')
            self.proxies.assign(np.asarray(init_proxies, np.float32) * PROXY_NORM)
        self.t_px = tf.keras.metrics.Mean(name='proxy_loss')
        self.t_loss = tf.keras.metrics.Mean(name='loss'); self.t_bce = tf.keras.metrics.Mean(name='bce'); self.t_at = tf.keras.metrics.Mean(name='at_loss'); self.t_hd = tf.keras.metrics.Mean(name='head_loss')
        self.t_acc = tf.keras.metrics.BinaryAccuracy(name='verification_accuracy'); self.t_auc = tf.keras.metrics.AUC(name='verification_auc')
    @property
    def metrics(self): return [self.t_loss, self.t_bce, self.t_px, self.t_at, self.t_hd, self.t_acc, self.t_auc]
    def proxy_loss(self, E, lab):      # FPP04L code (shared cross-sensor finger proxies)
        W = tf.math.l2_normalize(self.proxies, axis=1); cos = tf.matmul(E, W, transpose_b=True); onehot = tf.one_hot(lab, self.n_classes, dtype=tf.float32)
        if self.kind == 'bce_cosface':
            logits = COSFACE_S * (cos - COSFACE_M * onehot); return tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(onehot, logits))
        pos = tf.exp(-PA_ALPHA * (cos - PA_DELTA)) * onehot; neg = tf.exp(PA_ALPHA * (cos + PA_DELTA)) * (1.0 - onehot)
        with_pos = tf.cast(tf.reduce_sum(onehot, 0) > 0, tf.float32)
        lp = tf.reduce_sum(tf.math.log1p(tf.reduce_sum(pos, 0)) * with_pos) / tf.maximum(tf.reduce_sum(with_pos), 1.0)
        return lp + tf.reduce_sum(tf.math.log1p(tf.reduce_sum(neg, 0))) / float(self.n_classes)
    def forward(self, x_uint8, training):
        x = tf.cast(x_uint8, tf.float32); rgb = tf.concat([x, x, x], -1)
        if self.pre is not None: rgb = self.pre(rgb)
        outs = self.multi(rgb, training=False); feats = list(outs[:-1])                     # backbone in inference mode (BatchNorm frozen), as in the encoder
        e = self.ebn(self.dense(self.drop(outs[-1], training=training)), training=training)
        return tf.math.l2_normalize(tf.cast(e, tf.float32), axis=1), feats
    def call(self, inputs, training=False):
        ea, _ = self.forward(inputs[0], training); eb, _ = self.forward(inputs[1], training)
        return self.calib(tf.reduce_sum(ea * eb, axis=1))
    def guidance(self, feats, H, w):
        la = tf.constant(0.0); lh = tf.constant(0.0)
        for k, F in enumerate(feats):
            Ha, Hm = pooled(H, F); la += at_loss(F, Ha, w)
            if self.heads: lh += head_loss(self.heads[k](F), Hm, w)
        return la / len(feats), lh / len(feats)
    def train_step(self, data):
        (xa, xb, ha, hb, wa, wb, la, lb), y = data; y = tf.cast(tf.reshape(y, [-1]), tf.float32)
        Ha = tf.cast(ha, tf.float32) / 255.0; Hb = tf.cast(hb, tf.float32) / 255.0; wa = tf.cast(wa, tf.float32); wb = tf.cast(wb, tf.float32)
        with tf.GradientTape() as tape:
            ea, fa = self.forward(xa, True); eb, fb = self.forward(xb, True)
            p = self.calib(tf.reduce_sum(ea * eb, axis=1)); l_bce = self.bce(y, p)
            a1, h1 = self.guidance(fa, Ha, wa); a2, h2 = self.guidance(fb, Hb, wb); l_at = 0.5 * (a1 + a2); l_hd = 0.5 * (h1 + h2)
            l_px = tf.constant(0.0) if self.kind == 'bce' else self.proxy_loss(tf.concat([ea, eb], 0), tf.cast(tf.concat([la, lb], 0), tf.int32))
            loss = l_bce + LAMBDA_PROXY * l_px + self.lam * (l_at + l_hd)
            scaled = self.optimizer.scale_loss(loss) if hasattr(self.optimizer, 'scale_loss') else loss
        tv = self.trainable_variables; gr = tape.gradient(scaled, tv); self.optimizer.apply_gradients([(g_, v_) for g_, v_ in zip(gr, tv) if g_ is not None])
        for m_, v_ in ((self.t_loss, loss), (self.t_bce, l_bce), (self.t_px, l_px), (self.t_at, l_at), (self.t_hd, l_hd)): m_.update_state(v_)
        self.t_acc.update_state(y, p); self.t_auc.update_state(y, p); return {m_.name: m_.result() for m_ in self.metrics}

class LambdaSchedule(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.model.lam.assign(guidance_lambda(epoch + 1))
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['guidance_lambda'] = float(self.model.lam.numpy())

def init_proxies(encoder, labels, n_classes):
    # FPP04L: mean L2-normalised embedding of each shared finger class (DB1 + DB2 training originals), L2-normalised
    idx = np.flatnonzero(labels >= 0); E = L.l2n(T.embed(encoder, IMG_MOLF[idx], EMBED_BATCH)); W = np.zeros((n_classes, EMBEDDING_DIM), np.float32)
    np.add.at(W, labels[idx], E); return L.l2n(W)

def build_guided_model(encoder, spec, n_classes=0, init_proxies_=None):
    m = GuidedSiamese(encoder, spec, n_classes, init_proxies_, name='guided_siamese')
    dummy = np.zeros((1, IMAGE_SIZE, IMAGE_SIZE, 1), np.uint8); m((dummy, dummy))
    for hd, F in zip(m.heads, m.forward(dummy, False)[1]): hd(F)
    m.built = True
    m.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=LR_FULL, weight_decay=WEIGHT_DECAY), jit_compile=JIT_COMPILE)
    # the training view must reproduce the deployed encoder exactly
    x = IMG['MOLF_DB1'][:16]; e1 = m.forward(tf.constant(x), False)[0].numpy(); e2 = L.l2n(T.embed(encoder, x, 16)); cs = float(np.min(np.sum(e1 * e2, axis=1)))
    if cs < 0.999: raise RuntimeError(f'training view differs from the encoder (min cosine {cs:.5f})')
    return m, cs

def attention_check(m, rep, label):
    # FingerJetFX only scores this: per guided layer, minutia/background activation-energy ratio and Pearson(energy, heatmap) inside the print (MOLF validation images)
    v = P.BalancedValidation(PROTO[rep][1], OFFSET).indices; v = v[HASMIN_MOLF[v] > 0]          # MOLF validation images (global index), with minutiae
    idx = np.sort(np.random.default_rng(int(PROTO[rep][0]['seed'])).choice(v, min(ATTENTION_CHECK_IMAGES, len(v)), replace=False))
    f = tf.function(lambda z: m.forward(z, False)[1], reduce_retracing=True); st = {n: {'ratio': [], 'corr': []} for n in m.guided}; maps = []
    for s in range(0, len(idx), 64):
        b = idx[s:s + 64]; feats = f(tf.constant(IMG_MOLF[b])); H = tf.constant(HM_MOLF[b].astype(np.float32) / 255.0); I = tf.constant(IMG_MOLF[b].astype(np.float32))
        for n, F in zip(m.guided, feats):
            k = IMAGE_SIZE // int(F.shape[1]); A_ = energy(F).numpy(); Tm = tf.nn.avg_pool2d(H, k, k, 'VALID')[..., 0].numpy(); FG = tf.nn.avg_pool2d(I, k, k, 'VALID')[..., 0].numpy() < FOREGROUND_GRAY
            for j in range(len(b)):
                a, t_, fg = A_[j], Tm[j], FG[j]
                if s == 0 and j < 4 and n == m.guided[0]: maps.append(a)
                if fg.sum() < 10 or t_[fg].max() <= 0: continue
                hot = fg & (t_ >= 0.3 * t_[fg].max()); cold = fg & (t_ <= 0.05 * t_[fg].max())
                if hot.any() and cold.any(): st[n]['ratio'].append(a[hot].mean() / max(a[cold].mean(), 1e-12))
                if a[fg].std() > 0 and t_[fg].std() > 0: st[n]['corr'].append(np.corrcoef(a[fg], t_[fg])[0, 1])
    rows = [{'arm': label, 'repetition': rep, 'layer': n, 'feature_size': int(GUIDED_SIZES[k_]), 'minutia_to_background_energy_ratio': float(np.mean(v['ratio'])) if v['ratio'] else np.nan,
             'energy_heatmap_pearson': float(np.mean(v['corr'])) if v['corr'] else np.nan, 'images': len(v['corr'])} for k_, (n, v) in enumerate(st.items())]
    return rows, np.stack(maps), idx[:4]
print('Guided model ready | arms:', BACKBONES)

## 4. One run = one (arm, repetition), initialised from the FPP04L winner of the same repetition

In [ ]:
def load_init(encoder, rep):
    src = base_run(rep)
    if not L.run_complete(src): raise FileNotFoundError(f'{src} not finished')
    encoder.load_weights(str(src / 'encoder.weights.h5'))
    E0 = L.load_npz(src / 'embeddings_float32.npz')['MOLF_DB1'][:32]; E1 = L.l2n(T.embed(encoder, IMG['MOLF_DB1'][:32], 32)); cs = float(np.mean(np.sum(L.l2n(E0) * E1, axis=1)))
    if cs < INIT_CHECK_MIN: raise RuntimeError(f'{rep}: reloaded start encoder ({BASE_ARM}) does not reproduce its embeddings (mean cosine {cs:.4f})')
    return {'init_from': L.rel(ws, src), 'init_check_cosine': round(cs, 6)}

def train_run(arm, rep):
    rd = KDIR / 'runs' / rep / arm
    if L.run_complete(rd): print(arm, rep, 'complete -> skipped'); return
    lrd = P.local_run_dir(KERNEL + ('_smoke' if QUICK_SMOKE_TEST else ''), rep, arm); spec = ARMS[arm]
    proto, Pp = PROTO[rep]; seed = proto['seed']; tf.keras.backend.clear_session(); gc.collect(); T.set_all_seeds(seed); t = L.timer()
    encoder, info = build_encoder(MODEL_NAME, 'full'); init = load_init(encoder, rep); params = T.count_params(encoder)
    ref_att = KDIR / 'runs' / rep / 'baseline_reference_attention.csv'; labels, n_cls = CLASSES[rep]
    W0 = init_proxies(encoder, labels, n_cls) if BASE_LOSS != 'bce' else None
    siamese, view_cos = build_guided_model(encoder, spec, n_cls, W0); init.update(training_loss=f'{BASE_LOSS} + minutiae guidance', proxies=int(n_cls))
    init.update(guided_layers=' | '.join(siamese.guided), training_view_min_cosine=round(view_cos, 6), guidance_at=spec['at'], guidance_head=spec['head'])
    if not ref_att.is_file():                                              # where the baseline itself looks (start weights), once per repetition
        r0, m0, i0 = attention_check(siamese, rep, BASELINE); ref_att.parent.mkdir(parents=True, exist_ok=True); pd.DataFrame(r0).to_csv(ref_att, index=False)
        np.savez_compressed(KDIR / 'runs' / rep / 'baseline_reference_attention_maps.npz', maps=m0, idx=i0)
    ssf = float(proto['manifest']['same_subject_impostor_fraction'])
    train_seq = GuidedPairSeq(labels, IMG_MOLF, P.MixedPairs(P.sources(Pp, P.TRAIN_SOURCES, OFFSET), PAIR_BATCH, True, seed, TRAIN_PAIRS_PER_EPOCH, ssf), AUG_MOLF if TRAIN_AUGMENTATION else None, AUG_P_ORIGINAL, seed)
    bv = P.BalancedValidation(Pp, OFFSET, max_pairs=VAL_PAIRS)
    LOG.add(step='train_start', arm=arm, repetition=rep, **init, train_composition=json.dumps(train_seq.s.composition), validation_pairs=json.dumps(bv.composition), **params, **info)
    print(f'{arm} {rep[:6]} | guided layers {siamese.guided} | training view = encoder (min cosine {view_cos:.5f})')
    ckpt = lrd / 'best_siamese.weights.h5'; hist_csv = lrd / 'training_history.csv'; fit_marker = lrd / 'fit_complete.json'
    if ckpt.is_file() and fit_marker.is_file():
        train_s = json.loads(fit_marker.read_text())['training_seconds']; print(arm, rep, 'fit finished earlier in this session -> evaluating the local checkpoint')
    else:
        cbs = [LambdaSchedule(), EpochClock(), BalancedEERCallback(encoder, bv, ckpt, hist_csv)]
        siamese.fit(train_seq, epochs=EPOCHS, callbacks=cbs, verbose=2); train_s = t(); fit_marker.write_text(json.dumps({'training_seconds': train_s}))
    siamese.load_weights(ckpt)
    hist = pd.read_csv(hist_csv); b_i, _ = P.BalancedEERStopping.replay(hist, EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR); best = hist.iloc[b_i]
    emb = {d: L.l2n(T.embed(encoder, IMG[d], EMBED_BATCH)) for d in L.DATASETS}
    encoder.save_weights(lrd / 'encoder.weights.h5'); np.savez_compressed(lrd / 'embeddings_float32.npz', **{d: e.astype(np.float32) for d, e in emb.items()})
    att_rows, maps, idx4 = attention_check(siamese, rep, arm); pd.DataFrame(att_rows).to_csv(lrd / 'attention_check.csv', index=False); np.savez_compressed(lrd / 'attention_maps.npz', maps=maps, idx=idx4)
    gpu_batch_ms, gpu_single_ms = T.time_encoder(encoder, IMG['MOLF_DB1'][:64], TIMING_REPEATS)
    extra = {'backbone': MODEL_NAME, 'policy': 'full', **init, 'training_augmentation': CONFIG['training_augmentation'], 'early_stopping': P.SELECTION_RULE, 'best_epoch': int(best.epoch), 'epochs_run': int(len(hist)),
             'val_balanced_eer_at_best': float(best.val_balanced_eer), 'val_balanced_auc_at_best': float(best.val_balanced_auc), 'train_auc_at_best': float(best.get('verification_auc', np.nan)),
             'training_seconds': train_s, 'seconds_per_epoch': float(hist['epoch_seconds'].mean()) if 'epoch_seconds' in hist else float('nan'), 'pair_batch': PAIR_BATCH,
             'embed_ms_per_image_gpu_batch64': gpu_batch_ms, 'embed_ms_single_image_gpu': gpu_single_ms, 'embed_ms_single_image_cpu': float('nan'), **params, **info}
    S = {n: (np.asarray(Pp[n]['label'], int), L.pair_scores(emb[da], emb[db], Pp[n]).astype(np.float64)) for n, (da, db, _) in P.PAIR_SETS.items()}
    rows, thr = P.evaluate(S, KERNEL, arm, rep, extra)
    (lrd / 'encoder_config.json').write_text(json.dumps({'backbone': MODEL_NAME, 'arm': arm, 'guidance': spec, 'policy': 'full', 'init_from': init['init_from'], 'guided_layers': siamese.guided,
                                                         'embedding_dim': EMBEDDING_DIM, 'image_size': IMAGE_SIZE, 'dropout': DROPOUT, 'learning_rate': LR_FULL,
                                                         'inference': 'image only (same architecture as FPP04C / FPP04L); heads and proxies discarded', 'training_loss': f'{BASE_LOSS} + minutiae guidance', 'start': BASE_ARM, **info}, indent=2, default=str))
    P.save_run(lrd, S, rows, thr, {'kernel': KERNEL, 'arm': arm, 'repetition': rep, 'seed': seed, **extra})
    n_files = P.publish_run(lrd, rd, skip=('fit_complete.json', 'best_siamese.weights.h5'))     # the training checkpoint holds the discarded heads
    P.fig_set_panel(f'{arm} {rep[:6]} (frozen validation-EER threshold)', {st: [(*P.group_scores(S, st), thr)] for st in P.PAPER_SETS}, FIG / 'runs' / rep / f'{arm}_panel.png')
    P.fig_training_balanced(hist, f'{arm} {rep[:6]} — stopping on source-balanced validation EER', FIG / 'runs' / rep / f'{arm}_training.png')
    v = [r_ for r_ in rows if r_['set'] == P.SELECTION_SET][0]
    LOG.add(step='train_done', arm=arm, repetition=rep, status='ok', minutes=round(t() / 60, 1), best_epoch=int(best.epoch), files_to_drive=n_files,
            val_balanced_eer=round(100 * v['eer'], 2), val_balanced_auc=round(100 * v['roc_auc'], 2), val_balanced_f1=round(100 * v['f1_macro'], 2))
    del siamese, encoder, emb; gc.collect()

## 5. Train every arm for every repetition

In [ ]:
for b in ARMS_TO_RUN:
    for rep in REPS_TO_RUN:
        try: train_run(b, rep)
        except (tf.errors.ResourceExhaustedError, tf.errors.InternalError) as exc:
            LOG.add(step='train_failed', arm=b, repetition=rep, status='gpu_error', error=repr(exc)[:400]); print('GPU error', b, rep, repr(exc)[:200]); tf.keras.backend.clear_session(); gc.collect()
        except Exception as exc:
            LOG.add(step='train_failed', arm=b, repetition=rep, status='error', error=repr(exc)[:400]); print('FAILED', b, rep, exc)

## 6. Decision on VALIDATION only (FPP04L winner + arms); then time, test and FingerDOS for every candidate (reported, not used)

In [ ]:
RES, HIST, MISSING = [], [], []
for a in BACKBONES:
    for r in REPS_TO_RUN:
        rd = KDIR / 'runs' / r / a
        if L.run_complete(rd):
            RES.append(pd.read_csv(rd / 'metrics.csv')); h = pd.read_csv(rd / 'training_history.csv'); h.insert(0, 'arm', a); h.insert(1, 'repetition', r); HIST.append(h)
        else: MISSING.append((a, r))
if MISSING: print('Not finished yet:', MISSING)
if not RES: raise RuntimeError('no finished run: run section 5 first')
results = pd.concat(RES, ignore_index=True); results['kernel'] = results['arm']; histories = pd.concat(HIST, ignore_index=True)
complete = [a for a in BACKBONES if all(L.run_complete(KDIR / 'runs' / r / a) for r in REPS_TO_RUN)]
ref = prev_results[(prev_results.kernel == prev_winner) & prev_results.repetition.isin(REPS_TO_RUN)].copy(); ref['arm'] = BASELINE; ref['kernel'] = BASELINE   # section 1b rows: re-scored, not re-run
ALLR = pd.concat([ref, results], ignore_index=True); CANDIDATES = [BASELINE] + complete

# ---- 6.1 decision: validation only
WINNER, decision = P.decide(ALLR, CANDIDATES)
val_table = P.validation_table(ALLR, CANDIDATES)
print('DECISION RULE:', P.DECISION_RULE); display(decision.round(3)); print('Validation (mean ± std over repetitions, %):'); display(val_table)
print('WINNER (validation only):', WINNER)

# ---- 6.2 time: training (per run) and embedding (GPU, per run); inference network identical to FPP04C / FPP04L -> the fair CPU comparison applies unchanged
tcols = ['training_seconds', 'best_epoch', 'epochs_run', 'embed_ms_single_image_gpu', 'embed_ms_per_image_gpu_batch64']
tsrc = pd.concat([pd.read_csv(base_run(r) / 'metrics.csv').assign(kernel=BASELINE) for r in REPS_TO_RUN] + [results], ignore_index=True)
tsrc = tsrc[tsrc.set == P.SELECTION_SET]; timing = tsrc.groupby('kernel')[[c_ for c_ in tcols if c_ in tsrc.columns]].agg(['mean', 'std']).reindex(CANDIDATES)
timing.columns = [f'{a}_{b}' for a, b in timing.columns]; print('Training and GPU embedding time (mean / std over repetitions):'); display(timing.round(3))
cpu_fair = FPP04C_DIR / 'cpu_timing' / 'FPP04C_fair_timing_comparison.csv'
fair = pd.read_csv(cpu_fair) if cpu_fair.is_file() else pd.DataFrame()
if len(fair): print('Fair CPU comparison (1 thread, same machine; measured on FPP04C, identical network):'); display(fair[fair.method.str.contains('EfficientNet|BOZORTH3')].round(4))

# ---- 6.3 measured on test and FingerDOS for every candidate — reported only, NOT used for the decision
REPORT_SETS = ['MOLF test (balanced)', 'DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test']
paper_all, snum_all = P.paper_table(ALLR, CANDIDATES, sets=['MOLF validation (balanced)'] + REPORT_SETS)
test_table = P.wide_table(snum_all, CANDIDATES, sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], metrics_=('roc_auc', 'eer', 'fmr', 'fnmr', 'f1_macro'))
print('Test and FingerDOS (frozen validation-EER threshold; reported, not used for the decision):'); display(test_table)
paired = pd.concat([P.paired_differences(ALLR, a, [BASELINE]) for a in complete], ignore_index=True) if complete else pd.DataFrame()
if len(paired): print(f'Paired differences arm - {BASELINE} (points):'); display(paired[paired.set.isin(['MOLF validation (balanced)'] + list(P.VAL_SOURCES)) & paired.metric.isin(['eer', 'roc_auc', 'f1_macro'])].round(3))

# ---- 6.4 attention diagnostic (where the early layers look) — interpretation only
ATT = [pd.read_csv(KDIR / 'runs' / r / 'baseline_reference_attention.csv') for r in REPS_TO_RUN if (KDIR / 'runs' / r / 'baseline_reference_attention.csv').is_file()]
ATT += [pd.read_csv(KDIR / 'runs' / r / a / 'attention_check.csv') for a in complete for r in REPS_TO_RUN]
attention = pd.concat(ATT, ignore_index=True) if ATT else pd.DataFrame()
att_summary = attention.groupby(['arm', 'feature_size'])[['minutia_to_background_energy_ratio', 'energy_heatmap_pearson']].agg(['mean', 'std']).round(3) if len(attention) else pd.DataFrame()
if len(att_summary): att_summary.columns = [f'{a}_{b}' for a, b in att_summary.columns]
print('Attention diagnostic (MOLF validation images; FingerJetFX only scores it):'); display(att_summary)

## 6b. Bias check of the threshold: 5 calibration hold-out subjects vs validation (MOLF test, balanced)

In [ ]:
CB = []
for b in CANDIDATES:
    for r in REPS_TO_RUN:
        rd = base_run(r) if b == BASELINE else (KDIR / 'runs' / r / b); proto, Pp = PROTO[r]
        c = P.calibration_check(P.load_scores(rd), L.load_npz(rd / 'embeddings_float32.npz'), P.calibration_pairs(proto)); CB.append({'arm': b, 'repetition': r, **c})
calib = pd.DataFrame(CB); cs = calib.drop(columns='repetition').groupby('arm').agg(['mean', 'std'])
calib_table = pd.DataFrame({'backbone': cs.index,
    'threshold validation': [f"{a:.3f} ± {b_:.3f}" for a, b_ in zip(cs[('threshold_validation', 'mean')], cs[('threshold_validation', 'std')])],
    'threshold calibration (5 subj.)': [f"{a:.3f} ± {b_:.3f}" for a, b_ in zip(cs[('threshold_calibration', 'mean')], cs[('threshold_calibration', 'std')])],
    **{f'test {m.upper()} {tag} (%)': (100 * cs[(f'test_{m}_{tag}', 'mean')]).round(2).to_numpy() for m in ('fmr', 'fnmr', 'f1') for tag in ('validation', 'calibration')}}).reset_index(drop=True)
print('Bias check (paper: supplementary): same operating point from 5 never-used subjects -> the validation threshold is not biased by reuse for selection'); display(calib_table)

## 7. Figures

In [ ]:
MR = {BASELINE: [base_run(r) for r in REPS_TO_RUN], **{a: [KDIR / 'runs' / r / a for r in REPS_TO_RUN] for a in complete}}
P.standard_figures(FIG, MR, sets=P.PAPER_SETS)
P.fig_metric_bars(snum_all, CANDIDATES, FIG / 'metric_bars_validation.png', sets=['MOLF validation (balanced)'], title='Validation (decision set): metrics per candidate')
P.fig_metric_bars(snum_all, CANDIDATES, FIG / 'metric_bars_test_reported.png', sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], title='Test and FingerDOS (reported, not used for the decision)')
fig, ax = plt.subplots(figsize=(8, 4))
for k, a in enumerate(complete):
    h = histories[histories.arm == a].groupby('epoch').val_balanced_eer.mean(); ax.plot(h.index, 100 * h.values, color=P.COLORS[k % 10], label=a)
ax.set(xlabel='Epoch', ylabel='validation EER, source-balanced (%)', title='Training curves (mean over repetitions)'); ax.legend(fontsize=8); ax.grid(alpha=.3); P._save(fig, FIG / 'training_curves.png')
if len(attention):
    fig, axs = plt.subplots(1, 2, figsize=(13, 3.5))
    for ax, col, lab in zip(axs, ('minutia_to_background_energy_ratio', 'energy_heatmap_pearson'), ('Minutia / background energy ratio', 'Pearson(energy, FingerJetFX heatmap)')):
        q = attention.groupby(['arm', 'feature_size'])[col].agg(['mean', 'std']).reset_index(); arms_ = [a for a in CANDIDATES if a in set(q.arm)]
        for k, fs in enumerate(GUIDED_SIZES):
            z = q[q.feature_size == fs].set_index('arm').reindex(arms_); ax.barh(np.arange(len(arms_)) + 0.4 * k, z['mean'], 0.38, xerr=z['std'], label=f'{fs}x{fs}', color=P.COLORS[k])
        ax.set_yticks(np.arange(len(arms_)) + 0.2); ax.set_yticklabels(arms_, fontsize=8); ax.set_xlabel(lab); ax.legend(fontsize=8); ax.grid(alpha=.3, axis='x')
    fig.suptitle('Where do the early layers look? (MOLF validation images, image-only input)', fontsize=10); P._save(fig, FIG / 'attention_check.png')
    r0 = REPS_TO_RUN[0]; panels = [(BASELINE, KDIR / 'runs' / r0 / 'baseline_reference_attention_maps.npz')] + [(a, KDIR / 'runs' / r0 / a / 'attention_maps.npz') for a in complete]
    panels = [(a, p_) for a, p_ in panels if p_.is_file()]; z0 = np.load(panels[0][1]); idx0 = z0['idx']
    fig, axs = plt.subplots(len(panels) + 1, 4, figsize=(12, 3 * (len(panels) + 1)), squeeze=False)
    for j, i in enumerate(idx0):
        axs[0, j].imshow(IMG_MOLF[i, ..., 0], cmap='gray'); axs[0, j].imshow(HM_MOLF[i, ..., 0], cmap='jet', alpha=.35); axs[0, j].set_title(f'image {i} + FingerJetFX heatmap', fontsize=8); axs[0, j].axis('off')
    for r_, (a, p_) in enumerate(panels, 1):
        mp = np.load(p_)['maps']
        for j in range(min(4, len(mp))): axs[r_, j].imshow(mp[j], cmap='magma'); axs[r_, j].set_title(a, fontsize=8); axs[r_, j].axis('off')
    fig.suptitle(f'Stride-4 activation energy, MOLF validation images ({r0[:6]})', fontsize=10); P._save(fig, FIG / 'early_layer_attention_maps.png')
print('figures in', FIG)

## 8. Manifest and workbook

In [ ]:
if not QUICK_SMOKE_TEST and len(complete) == len(BACKBONES):
    files = ('encoder.weights.h5', 'encoder_config.json', 'embeddings_float32.npz', 'thresholds.json', 'pair_scores.npz')
    wdir = base_run if WINNER == BASELINE else (lambda r: KDIR / 'runs' / r / WINNER)
    manifest = {'kernel': KERNEL, 'kernel_name': P.FPP_KERNELS[KERNEL], 'backbone': MODEL_NAME, 'winning_arm': WINNER, 'winner_is_baseline': WINNER == BASELINE, 'previous_winner': prev_winner, 'previous_decision': prev_decision.to_dict('records'), 'training_loss': f'{BASE_LOSS} + minutiae guidance',
                'decision_rule': P.DECISION_RULE, 'decision': decision.to_dict('records'), 'threshold_rule': P.THRESHOLD_RULE, 'selection_rule': P.SELECTION_RULE,
                'note': (f'minutiae-guided early attention does not improve validation: {BASELINE} stays the image model' if WINNER == BASELINE else f'{WINNER} improves the source-balanced validation EER over {BASELINE}'),
                'files': {r: {k: L.rel(ws, wdir(r) / k) for k in files} for r in REPS_TO_RUN}, 'fpp_lib_version': P.VERSION}
    (KDIR / 'winner_manifest.json').write_text(json.dumps(manifest, indent=2, default=str)); print('winner_manifest.json written | winner:', WINNER)
SHEETS = {'FPP04L decision (re-scored)': prev_decision, 'Decision (validation)': decision, 'Validation table': val_table, 'Time': timing.reset_index(), 'Fair CPU timing (FPP04C)': fair,
          'Test+FingerDOS (reported)': test_table, 'All sets (mean ± std)': paper_all, 'Paired vs baseline': paired, 'Attention diagnostic': attention,
          'Attention summary': att_summary.reset_index() if len(att_summary) else att_summary, 'Heatmap alignment': alignment, 'Calibration Bias Check': calib_table,
          'Calibration (per rep)': calib, 'Results (per rep)': P.to_percent(ALLR), 'Training history': histories, 'Config': pd.DataFrame([{k: json.dumps(v, default=str) for k, v in CONFIG.items()}]).T.reset_index(),
          'Run log': LOG.frame()}
L.write_workbook(KDIR / 'FPP05_Minutiae_Guided_Attention_Results.xlsx', SHEETS)
for name, df in (('decision', decision), ('validation_table', val_table), ('time', timing.reset_index()), ('test_fingerdos_reported', test_table), ('paired_vs_baseline', paired),
                 ('attention', attention), ('calibration_bias_check', calib_table), ('results_per_repetition', P.to_percent(ALLR))):
    df.to_csv(KDIR / f'FPP05_{name}.csv', index=False)
print('FPP05 complete | winner on validation:', WINNER)